# 08 — First training on the reviewed Luna dataset

Goal: measure what the **new dataset alone** brings to the nano model, without adding the Infactory images to training.

We compare the pretrained COCO nano, our previous v002 nano (10 epochs on Infactory) and the nano trained here. Same 640 resolution, same test images, same thresholds.

This is a private pilot: the redistribution rights for the video are not established. This notebook publishes nothing and launches no paid Luna call.

In [ ]:
import csv
import gc
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("YOLO_AUTOINSTALL", "false")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_ROOT = PROJECT_ROOT / "data/processed/luna_v001"
RUN_ROOT = PROJECT_ROOT / "models/checkpoints/baseline_runs/yolo26n_luna_v001_20260928_pilot"
REPORT_PATH = RUN_ROOT / "report.json"
build = json.loads((DATA_ROOT / "build_report.json").read_text())
with (DATA_ROOT / "manifest.csv").open() as stream:
    manifest = list(csv.DictReader(stream))
print("Project:", PROJECT_ROOT)
print("Results:", RUN_ROOT)

## 1. What goes into the experiment

The compilation contains several matches. We identified **10 probable groups** by inspecting the scores, the jerseys and the pitches. All clips and replays of the same group stay in a single split. These are visually inferred identities, not verified match identifiers.

The possible allocations with 6 train, 2 validation and 2 test groups are compared to approach 70/15/15 in images, positives and negatives. No model prediction is involved in this choice.

At the nine transitions between matches, a small margin removes the cross-faded frames that mix two matches. The original human decisions are left unchanged.

In [ ]:
lines = ["| Split | Images | With ball | Without ball | Match groups |",
         "|---|---:|---:|---:|---|"]
for split, row in build["splits"].items():
    lines.append(f"| {split} | {row['images']} | {row['positive']} | {row['negative']} | {', '.join(row['matches'])} |")
display(Markdown("\n".join(lines)))
print("Exclusions:", build["excluded"])
print("Accepted images without a Luna response:", build["images_without_luna_answer_submitted_empty"])
print("Storage:", build["image_storage"])
# Check that groups are exclusive to a single split.
owners = {}
for row in manifest:
    owners.setdefault(row["match_id"], set()).add(row["split"])
assert all(len(splits) == 1 for splits in owners.values())
assert len(manifest) == build["images"]
print("No group shared between splits.")

**Watch out for negative quality:** 49 kept images had no Luna response and were submitted empty during the requested global acceptance. An empty box is therefore not proof that the ball is absent. This can bias the metrics; these cases remain traceable in the manifest.

The JPEGs are linked to the images already present, without copying their data. Do not modify these files in place.

In [ ]:
overview = PROJECT_ROOT / "models/reports/luna_scene_overview.jpg"
if overview.exists():
    display(DisplayImage(filename=str(overview), width=900))

## 2. Parameters and launch

For this pilot: Apple MPS GPU, batch 2, no image caching, 640 px, AdamW. Initialized from the COCO nano, **not from the Infactory checkpoint**.

The first run uses a time limit of 18 minutes and early stopping after 5 epochs without improvement. In Ultralytics, a time limit can recompute the number of epochs: the number actually completed is shown below.

The next cell is disabled by default so the work is not re-run by accident. For a new experiment, change RUN_ROOT to an unused name and enable RUN_TRAINING.

In [ ]:
RUN_TRAINING = False
EPOCHS = 15
IMAGE_SIZE = 640
BATCH = 2
DEVICE = "mps"

if RUN_TRAINING:
    import subprocess
    import time
    if RUN_ROOT.exists():
        raise FileExistsError("Choose an unused RUN_ROOT: no existing result will be replaced.")
    log_path = PROJECT_ROOT / "models/reports" / (RUN_ROOT.name + ".log")
    command = [sys.executable, "-u", str(PROJECT_ROOT / "scripts/train_luna_baseline.py"),
               "--epochs", str(EPOCHS), "--imgsz", str(IMAGE_SIZE),
               "--batch", str(BATCH), "--device", DEVICE, "--output", str(RUN_ROOT)]
    status = display(Markdown("Starting…"), display_id=True)
    with log_path.open("w") as log:
        process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            tail = log_path.read_text(errors="replace").splitlines()[-5:]
            status.update(Markdown("~~~text\n" + "\n".join(tail) + "\n~~~"))
            time.sleep(2)
    if process.returncode:
        raise RuntimeError(f"Failed: see {log_path}")
    status.update(Markdown("Training and evaluations finished."))

In [ ]:
report = json.loads(REPORT_PATH.read_text())
display(Markdown(f"**Status:** {report['status']} — **Epochs logged:** {report.get('epochs_logged', report.get('epochs_completed', 'in progress'))}"))
print("The time limit can stop the last epoch partway through the pass over the images.")
if report["status"] != "complete":
    print("The report is partial; re-run this cell after it finishes.")
print("Configuration:", report["training"])

## 3. Comparison under identical conditions

The validation split is used to choose best.pt. The test split is used **only after training and this choice**, to compare the three models.

Precision, recall, F1 and TP/FP/FN are computed with confidence ≥ 0.25 and IoU ≥ 0.50. A TP is a box matched to an annotation; an FP is an unmatched box; an FN is a missed annotation.

The mAP uses the whole precision/recall curve, not just the 0.25 threshold. mAP50–95 also requires more precisely localized boxes. For the COCO model, only the "sports ball" class is compared with the "ball" annotations. No comparison between the metrics of old and new splits.

In [ ]:
names = {"pretrained": "COCO nano", "v002": "Previous Infactory nano", "luna": "New Luna nano"}
def metrics_table(results):
    lines = ["| Model | mAP50 | mAP50–95 | Precision | Recall | F1 | TP | FP | FN | FP on negatives |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for name, row in results.items():
        fixed = row["fixed"]
        values = [row["map50"], row["map50_95"], fixed["precision"], fixed["recall"], fixed["f1"]]
        formatted = " | ".join(f"{v:.1%}" for v in values)
        counts = " | ".join(str(fixed.get(k, 0)) for k in ["tp", "fp", "fn", "fp_on_negatives"])
        lines.append(f"| {names[name]} | {formatted} | {counts} |")
    display(Markdown("\n".join(lines)))

display(Markdown("### Validation — 176 images"))
metrics_table(report["validation"])
display(Markdown("### Final test — 176 images from two other matches"))
metrics_table(report["test"])

## 4. Learning curves

A drop in the train losses accompanied by a lasting rise in the validation losses and a degradation of the metrics may signal overfitting. A single uptick is not enough to conclude.

The train/validation values are not perfectly comparable: augmentations and computation conditions differ. Look at the trends over several epochs and at the validation performance together.

In [ ]:
with (RUN_ROOT / "train/results.csv").open() as stream:
    history = [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(stream)]
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, loss in zip(axes.flat[:3], ["box_loss", "cls_loss", "l1_loss"]):
    for split in ["train", "val"]:
        key = f"{split}/{loss}"
        if key in history[0]:
            ax.plot(epochs, [row[key] for row in history], marker=".", label=split)
    ax.set(title=loss, xlabel="Epoch", ylabel="Loss")
    ax.legend()
    ax.grid(alpha=.25)
ax = axes.flat[3]
for key, label in [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50–95")]:
    ax.plot(epochs, [row[key] for row in history], marker=".", label=label)
ax.set(title="Validation", xlabel="Epoch", ylabel="mAP", ylim=(0, 1))
ax.legend()
ax.grid(alpha=.25)
fig.suptitle("Training on the new dataset only")
fig.tight_layout()
curves_path = RUN_ROOT / "learning_curves.png"
fig.savefig(curves_path, dpi=120)
plt.close(fig)
display(DisplayImage(filename=str(curves_path)))
display(Markdown(f"{len(history)} epochs observed. This short pilot alone does not allow a conclusion about overfitting."))

## 5. Confusion matrices at the threshold used

For single-class detection, the matrix is not that of a classic classification. The "background/background" cell has no natural number of true negatives: there are infinitely many areas without a ball.

Here the rows are the predictions and the columns the annotations. The values come exactly from the TP/FP/FN of the table at confidence 0.25 and IoU 0.50. The automatic Ultralytics matrices may use another threshold; they are not used below.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.7))
for ax, (name, row) in zip(axes, report["test"].items()):
    fixed = row["fixed"]
    counts = np.array([[fixed.get("tp", 0), fixed.get("fp", 0)], [fixed.get("fn", 0), np.nan]])
    ax.imshow(np.nan_to_num(counts), cmap="Blues", vmin=0, vmax=138)
    for i, j in np.ndindex(2, 2):
        ax.text(j, i, "N/A" if np.isnan(counts[i, j]) else str(int(counts[i, j])),
                ha="center", va="center", color="black", fontsize=13)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Ball", "Background"],
           yticklabels=["Ball", "Background"], xlabel="Annotation", ylabel="Prediction", title=names[name])
fig.tight_layout()
matrix_path = RUN_ROOT / "fixed_confusion_matrices.png"
fig.savefig(matrix_path, dpi=120)
plt.close(fig)
display(DisplayImage(filename=str(matrix_path)))

## 6. Precision/recall curves of the new model

The PR curve varies the confidence threshold; the F1 curve shows the effect of this threshold on the precision/recall trade-off. The test split remains a report, not a tool for tuning the model repeatedly.

In [ ]:
for filename in ["BoxPR_curve.png", "BoxF1_curve.png"]:
    path = RUN_ROOT / "evaluation/test_luna" / filename
    if path.exists():
        display(DisplayImage(filename=str(path), width=800))

## 7. Before/after inspection, without overloading memory

Before = previous Infactory nano; after = new nano trained on Luna. The **green** outline is the reviewed annotation, the **red** one the prediction at confidence ≥ 0.25. The text is placed outside the boxes.

The 176 test images are processed one by one on CPU, and only the coordinates are kept. The gallery shows at most three examples per category, with diverse matches when possible. No copy of the photos is created.

"Corrected": errors before, none after. "Regression": correct before, errors after. "Still wrong" can also represent a localization that is too imprecise, even if the ball is visually found.

In [ ]:
RUN_VISUAL_INSPECTION = True
ERROR_PATH = RUN_ROOT / "error_analysis_validation.json"

if RUN_VISUAL_INSPECTION and report["status"] == "complete" and not ERROR_PATH.exists():
    import torch
    from ultralytics import YOLO
    from ultralytics.utils import ops

    from pitchlens.training.diagnostics import (
        change_category,
        quiet_ultralytics_progress,
    )
    from pitchlens.training.evaluate import BallValidator

    class GalleryValidator(BallValidator):
        """Capture the exact predictions used by the metric table, one frame at a time."""
        def init_metrics(self, model):
            super().init_metrics(model)
            self.gallery = {}

        def update_metrics(self, preds, batch):
            for index, pred in enumerate(preds):
                target = self._prepare_batch(index, batch)
                selected = pred["conf"] >= .25
                filtered = {key: value[selected] for key, value in pred.items()}
                tp = int(self._process_batch(filtered, target)["tp"][:, 0].sum())
                boxes = ops.scale_boxes(target["imgsz"], filtered["bboxes"].clone(),
                                        target["ori_shape"], target["ratio_pad"]).cpu().numpy()
                h, w = target["ori_shape"]
                self.gallery[Path(target["im_file"]).stem] = {
                    "boxes": (boxes / np.array([w, h, w, h])).tolist(),
                    "confidence": filtered["conf"].cpu().tolist(),
                    "counts": {"tp": tp, "fp": len(boxes)-tp, "fn": len(target["cls"])-tp},
                }
            super().update_metrics(preds, batch)

        def finalize_metrics(self):
            super().finalize_metrics()
            self.metrics.gallery = self.gallery

    torch.set_num_threads(4)
    test_rows = [row for row in manifest if row["split"] == "test"]
    analysis = []
    for row in test_rows:
        label_path = DATA_ROOT / "test/labels" / (row["item_id"] + ".txt")
        ground_truth = []
        for line in label_path.read_text().splitlines():
            cls, x, y, w, h = map(float, line.split())
            ground_truth.append([x-w/2, y-h/2, x+w/2, y+h/2])
        analysis.append({"item_id": row["item_id"], "match_id": row["match_id"],
                         "ground_truth": ground_truth})
    for key, label, weight in [("before", "v002", report["previous_weights"]),
                               ("after", "luna", report["best_weights"])]:
        model = YOLO(weight)
        with quiet_ultralytics_progress():
            metrics = model.val(validator=GalleryValidator, data=str(DATA_ROOT / "dataset.yaml"),
                                split="test", imgsz=640, device="cpu", batch=1, workers=0,
                                conf=.001, iou=.7, max_det=100, plots=False, verbose=False,
                                project=str(RUN_ROOT / "evaluation"), name="gallery_"+label, exist_ok=True)
        for count in ["tp", "fp", "fn"]:
            assert metrics.fixed_threshold[count] == report["test"][label]["fixed"][count]
        for row in analysis:
            prediction = metrics.gallery[row["item_id"]]
            row[key] = prediction["boxes"]
            row[key+"_confidence"] = prediction["confidence"]
            row[key+"_counts"] = prediction["counts"]
        del model, metrics
        gc.collect()
    for row in analysis:
        row["category"] = change_category(row["before_counts"], row["after_counts"])
    ERROR_PATH.write_text(json.dumps(analysis, indent=2))
    print("Inspection computed: the totals match the table exactly.")
elif ERROR_PATH.exists():
    print("Inspection already computed: no inference re-run.")

In [ ]:
if ERROR_PATH.exists():
    from collections import Counter

    from PIL import Image as PILImage
    analysis = json.loads(ERROR_PATH.read_text())
    print("Categories over the 176 images:", dict(Counter(row["category"] for row in analysis)))
    lines = ["| Test match | Model | Images | TP | FP | FN | F1 |",
             "|---|---|---:|---:|---:|---:|---:|"]
    for match in sorted({row["match_id"] for row in analysis}):
        rows = [row for row in analysis if row["match_id"] == match]
        for key, label in [("before", "Previous"), ("after", "New")]:
            counts = Counter()
            for row in rows:
                counts.update(row[key + "_counts"])
            tp, fp, fn = (counts[k] for k in ["tp", "fp", "fn"])
            f1 = 2*tp / (2*tp+fp+fn) if 2*tp+fp+fn else 0
            lines.append(f"| {match} | {label} | {len(rows)} | {tp} | {fp} | {fn} | {f1:.1%} |")
    display(Markdown("\n".join(lines)))

    def diverse_examples(rows, limit=3):
        chosen = []
        for match in sorted({row["match_id"] for row in rows}):
            chosen.append(next(row for row in rows if row["match_id"] == match))
        for row in rows:
            if len(chosen) >= limit:
                break
            if row not in chosen:
                chosen.append(row)
        return chosen[:limit]

    for category, title in [("corrected", "Errors corrected"),
                            ("regressed", "Regressions"),
                            ("still_wrong", "Remaining errors")]:
        selected = diverse_examples([r for r in analysis if r["category"] == category])
        if not selected:
            display(Markdown(f"### {title}\nNone in this test at the chosen threshold."))
            continue
        fig, axes = plt.subplots(len(selected), 2, figsize=(12, 3.5*len(selected)), squeeze=False)
        for i, row in enumerate(selected):
            with PILImage.open(DATA_ROOT / "test/images" / (row["item_id"] + ".jpg")) as original:
                original.thumbnail((800, 450))
                pixels = np.asarray(original).copy()
            h, w = pixels.shape[:2]
            for j, (key, label) in enumerate([("before", "Before — Infactory"), ("after", "After — Luna")]):
                ax = axes[i, j]
                ax.imshow(pixels)
                for box in row["ground_truth"]:
                    x1, y1, x2, y2 = np.array(box)*[w, h, w, h]
                    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=1))
                for box, confidence in zip(row[key], row[key+"_confidence"]):
                    x1, y1, x2, y2 = np.array(box)*[w, h, w, h]
                    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=.8))
                    ax.text(x1, max(9, y1-5), f"{confidence:.0%}", fontsize=7, color="red",
                            bbox={"facecolor": "white", "alpha": .8, "pad": .5, "edgecolor": "none"})
                counts = row[key+"_counts"]
                ax.set_title(f"{label} · {row['match_id']} · TP {counts['tp']} / FP {counts['fp']} / FN {counts['fn']}", fontsize=9)
                ax.axis("off")
        fig.suptitle(title + " — green: annotation, red: prediction")
        fig.tight_layout()
        gallery_path = RUN_ROOT / (category + "_gallery.png")
        fig.savefig(gallery_path, dpi=110)
        plt.close(fig)
        display(DisplayImage(filename=str(gallery_path)))

## 8. Camera decision benchmark: a single target

On the **same 176 test images**, apply a rule close to the intended use:

1. Reject proposals below the confidence threshold (25% by default).
2. Keep only the most confident box; in case of a tie, the first one.
3. Accept this target if its IoU with an annotated ball reaches 10% or 20%.

IoU = intersection / union: "20%" means IoU ≥ 0.20, not 20% of the area of a box. A wrong secondary box no longer penalizes the decision if the best one targets the ball. The right box is never chosen by consulting the annotation.

Precision = correct targets / selected targets. Recall = correct targets / images containing an annotated ball. Choosing a wrong target on a positive image counts as FP + FN, but as a single wrong decision. On an empty image, staying silent is correct; selecting a target is a false alarm.

**Limits:** this is not yet a test of temporal stability, latency or the gimbal. With several real balls, the labels do not say which one is in play: any annotated box can count as correct. Among the 38 negatives of the test split, 12 come from a missing Luna response accepted as empty: the alerts remain to be reviewed visually.

The threshold is fixed before reading the test results. To tune it, use the validation split and keep another independent test split. The cache only contains proposals ≥ 25%: a lower threshold requires a new inference.

In [ ]:
import hashlib

from pitchlens.training.camera_benchmark import benchmark_targets

CAMERA_CONFIDENCE = .25
CAMERA_IOUS = [.10, .20]
camera_rows = json.loads(ERROR_PATH.read_text())
camera_report = {
    "scope": "offline frame-level camera decision proxy, not temporal tracking",
    "confidence": CAMERA_CONFIDENCE,
    "selection": "highest confidence only; first wins exact ties",
    "matching": "IoU with any annotated ball; active ball not identified",
    "cached_min_confidence": .25,
    "cache_sha256": hashlib.sha256(ERROR_PATH.read_bytes()).hexdigest(),
    "test_note": "Two inferred matches; 12 empty labels without a Luna response",
    "results": {},
}
lines = ["| Model | Min. IoU | Correct targets | Precision | Recall | F1 | Wrong targets on positives | Alerts on negatives |",
         "|---|---:|---:|---:|---:|---:|---:|---:|"]
for iou in CAMERA_IOUS:
    for key, label in [("before", "Previous Infactory"), ("after", "New Luna")]:
        result = benchmark_targets(camera_rows, key, confidence=CAMERA_CONFIDENCE, matching_iou=iou)
        camera_report["results"][f"{key}_iou{iou:.2f}"] = result
        metrics = result["metrics"]
        lines.append(
            f"| {label} | {iou:.0%} | {metrics['tp']}/{metrics['positive_images']} | "
            f"{metrics['precision']:.1%} | {metrics['recall']:.1%} | {metrics['f1']:.1%} | "
            f"{metrics.get('wrong_target', 0)} | {metrics.get('false_alarm', 0)}/{metrics['negative_images']} |"
        )
display(Markdown("\n".join(lines)))
(RUN_ROOT / "camera_benchmark.json").write_text(json.dumps(camera_report, ensure_ascii=False, indent=2))
display(Markdown(
    "At 25% confidence, the 10% and 20% IoU thresholds give the same results here. "
    "The new model is not better under this operational rule: "
    "the classic metrics and the camera decision measure different things."
))

## 9. Reading the first pilot

On the two test matches, the new nano goes from **14.9% to 33.6% mAP50** compared with our previous Infactory model. Its recall goes from **18.8% to 37.0%**, but false positives increase from **60 to 134** at the 0.25 threshold. Precision drops from 30.2% to 27.6%. F1 rises from 23.2% to 31.6%.

The unadapted COCO nano gets 24.8% mAP50 and 29.1% F1: the new model improves on these scores, but remains much less selective at the fixed threshold. We should therefore not conclude that it is already more reliable for steering the camera.

The best checkpoint chosen on validation reaches 53.0% mAP50 on that split, versus 33.6% on the test split: the other two matches remain difficult. This gap, over so few matches, does not by itself prove overfitting.

This test measures a change of dataset **and** of training, not only a change of model. The annotations, the diversity of matches and the amount of data can all influence the results. Fifteen complete epochs and one last partial one were recorded.

**Suggested next step:** check the false detections and the negatives without a Luna response, then add more matches with explicit identifiers. If the confidence threshold is adjusted, choose it on the validation split, not on the test split. Keep a new independent evaluation for the next choices.